# 05 — Met Office Weather DataHub (Land Observations)

Fetches Met Office **Land Observations** for each configured site:
1) `/nearest` by lat/lon (returns nearest stations including `geohash`)
2) `/{geohash}` for latest ~48h observations

Hardening included:
- rounds lat/lon to **2 decimals** (required)
- defensive geohash extraction
- drops malformed obs rows where the dict contains only `datetime`

Outputs:
- `outputs/05_metoffice_datahub_weather/raw/*.json`
- `outputs/05_metoffice_datahub_weather/derived/nearest_index.json`
- `outputs/05_metoffice_datahub_weather/manifest.json`


In [ ]:
import os, json, hashlib, time, random
from pathlib import Path
from datetime import datetime, timezone
import requests
import yaml

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def env(name: str):
    v = os.getenv(name)
    return v.strip() if isinstance(v, str) and v.strip() else None

def sha256_file(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()

def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')

def manifest_base(step: str, config_paths=None):
    m = {'step': step, 'created_at_utc': utc_now(), 'configs': [], 'requests': [], 'artifacts': []}
    for cp in (config_paths or []):
        if cp and cp.exists():
            m['configs'].append({'path': str(cp), 'sha256': sha256_file(cp)})
    return m

def add_artifact(m: dict, path: Path, kind: str, meta=None):
    if path.exists():
        item = {'kind': kind, 'path': str(path), 'sha256': sha256_file(path)}
        if meta:
            item['meta'] = meta
        m['artifacts'].append(item)

def record_request(m: dict, url: str, params: dict | None, r: requests.Response, out_path: Path):
    rec = {
        'ts_utc': utc_now(),
        'url': url,
        'params': params or {},
        'status': int(r.status_code),
        'ok': bool(r.ok),
        'content_type': r.headers.get('content-type')
    }
    if out_path.exists():
        rec['response_path'] = str(out_path)
        rec['response_sha256'] = sha256_file(out_path)
    m['requests'].append(rec)

def find_repo_root(start: Path) -> Path:
    cur = start
    for _ in range(12):
        if (cur/'configs').exists() and (cur/'notebooks').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start

CWD = Path.cwd().resolve()
ROOT = find_repo_root(CWD)
CONFIGS = ROOT/'configs'
OUTPUTS = ROOT/'outputs'

print('UTC now:', utc_now())
print('CWD:', CWD)
print('ROOT:', ROOT)
print('CONFIGS exists:', CONFIGS.exists())
print('OUTPUTS exists:', OUTPUTS.exists())


In [ ]:
cfg_path = CONFIGS/'run.yml'
if not cfg_path.exists():
    raise FileNotFoundError(f"configs/run.yml not found at: {cfg_path}")
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
def normalize_sites(cfg: dict):
    sites_norm = []
    sites = cfg.get('sites')
    if isinstance(sites, list):
        for s in sites:
            sid = s.get('id') or s.get('site_id') or s.get('name')
            sites_norm.append({
                'site_id': sid,
                'name': s.get('name'),
                'lat': s.get('lat'),
                'lon': s.get('lon'),
                'role': (s.get('role') or 'context')
            })
    elif isinstance(sites, dict):
        for k, v in sites.items():
            sites_norm.append({
                'site_id': k,
                'name': v.get('name'),
                'lat': v.get('lat'),
                'lon': v.get('lon'),
                'role': (v.get('role') or 'context')
            })
    else:
        raise RuntimeError("run.yml has no usable 'sites' (expected list or dict)")

    controls = cfg.get('controls')
    if isinstance(controls, dict):
        for k, v in controls.items():
            sites_norm.append({
                'site_id': k,
                'name': v.get('name'),
                'lat': v.get('lat'),
                'lon': v.get('lon'),
                'role': 'control'
            })

    usable = [s for s in sites_norm if s.get('site_id') and s.get('lat') is not None and s.get('lon') is not None]
    return usable
sites = normalize_sites(cfg)
print('Sites usable:', len(sites))
for s in sites:
    print('-', s['site_id'], s.get('name'), s.get('lat'), s.get('lon'), s.get('role'))


In [ ]:
APIKEY = env('MET_OFFICE_LAND_OBSERVATIONS') or env('MET_OFFICE_KEY') or env('METOFFICE_API_KEY')
if not APIKEY:
    raise RuntimeError('Missing Met Office API key. Set MET_OFFICE_LAND_OBSERVATIONS or MET_OFFICE_KEY/METOFFICE_API_KEY')

BASE_URL = 'https://data.hub.api.metoffice.gov.uk/observation-land/1'
SESSION = requests.Session()
SESSION.headers.update({'apikey': APIKEY})

def save_response(path: Path, r: requests.Response):
    ct = (r.headers.get('content-type') or '').lower()
    if 'application/json' in ct:
        try:
            write_json(path, r.json())
            return
        except Exception:
            pass
    write_json(path, {
        'status': 'nonjson_or_parse_error',
        'http_status': int(r.status_code),
        'content_type': r.headers.get('content-type'),
        'text': r.text[:20000]
    })

def get_json(url: str, params: dict | None, out_path: Path, manifest: dict):
    r = SESSION.get(url, params=params, timeout=30)
    save_response(out_path, r)
    record_request(manifest, url, params, r, out_path)
    return r

def extract_geohash(obj):
    if isinstance(obj, list) and obj:
        first = obj[0]
        if isinstance(first, dict) and isinstance(first.get('geohash'), str):
            return first.get('geohash')
    if isinstance(obj, dict) and isinstance(obj.get('geohash'), str):
        return obj.get('geohash')
    return None

def clean_observations(obj):
    if not isinstance(obj, list):
        return obj, 0
    cleaned = []
    dropped = 0
    for r in obj:
        if not isinstance(r, dict):
            dropped += 1
            continue
        if set(r.keys()) == {'datetime'}:
            dropped += 1
            continue
        cleaned.append(r)
    return cleaned, dropped


In [ ]:
step = '05_metoffice_datahub_weather'
out = OUTPUTS/step
raw = out/'raw'
derived = out/'derived'
raw.mkdir(parents=True, exist_ok=True)
derived.mkdir(parents=True, exist_ok=True)

manifest = manifest_base(step, [cfg_path])
nearest_index = []

for s in sites:
    site_id = s['site_id']
    lat = round(float(s['lat']), 2)
    lon = round(float(s['lon']), 2)

    nearest_url = f"{BASE_URL}/nearest"
    nearest_path = raw / f"metoffice_nearest_{site_id}.json"
    r1 = get_json(nearest_url, {'lat': lat, 'lon': lon}, nearest_path, manifest)

    try:
        j1 = json.loads(nearest_path.read_text(encoding='utf-8'))
    except Exception:
        j1 = None

    geohash = extract_geohash(j1)
    nearest_index.append({
        'site_id': site_id,
        'name': s.get('name'),
        'role': s.get('role'),
        'lat': lat,
        'lon': lon,
        'nearest_http_ok': bool(r1.ok),
        'nearest_http_status': int(r1.status_code),
        'geohash': geohash,
        'nearest_raw': str(nearest_path)
    })

    obs_path = raw / f"metoffice_obs_{site_id}.json"
    if geohash:
        obs_url = f"{BASE_URL}/{geohash}"
        _ = get_json(obs_url, None, obs_path, manifest)
        # clean malformed obs rows
        try:
            obs_obj = json.loads(obs_path.read_text(encoding='utf-8'))
            cleaned, dropped = clean_observations(obs_obj)
            if dropped:
                write_json(obs_path, cleaned)
                print(f"{site_id}: dropped {dropped} malformed obs rows")
        except Exception:
            pass
    else:
        write_json(obs_path, {'site_id': site_id, 'note': 'No geohash extracted from /nearest response; observations not fetched.'})

nearest_index_path = derived/'nearest_index.json'
write_json(nearest_index_path, nearest_index)

add_artifact(manifest, nearest_index_path, 'nearest_index')
for item in nearest_index:
    add_artifact(manifest, Path(item['nearest_raw']), 'raw_nearest', meta={'site_id': item['site_id']})
    add_artifact(manifest, raw / f"metoffice_obs_{item['site_id']}.json", 'raw_observations', meta={'site_id': item['site_id'], 'geohash': item.get('geohash')})

manifest_path = out/'manifest.json'
write_json(manifest_path, manifest)

print('Wrote:', nearest_index_path)
print('Wrote:', manifest_path)
print('Geohashes:')
for x in nearest_index:
    print('-', x['site_id'], '=>', x.get('geohash'), '(HTTP', x.get('nearest_http_status'), ')')
